# Imports

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
import pyspark
import re

# Parameters

In [0]:
SOURCE_PATH = "/Volumes/workspace/landing/general_files/edg_data"

FORMAT_MAP = {
                 "csv": "csv",
                 "json": "json",
                 "parquet": "parquet",
                 "avro": "avro",
                 "orc": "orc",
                 "txt": "text",
                 "xml": "xml",
             }

# Ingesting Data

In [0]:
def _list_files_recursive(path):
    """Recursively list all files under *path* (including subfolders)."""
    results = []
    for entry in dbutils.fs.ls(path):
        if entry.isDir():
            results.extend(_list_files_recursive(entry.path))
        else:
            results.append(entry)
    return results

files = _list_files_recursive(SOURCE_PATH)


for f in files:
    file_name = f.name
    file_path = f.path

    # Derive format and table name from the file extension
    ext = file_name.rsplit(".", 1)[-1].lower() if "." in file_name else ""
    read_format = FORMAT_MAP.get(ext)
    if read_format is None:
        print(f"Skipping unsupported file type: {file_name}")
        continue

    table_name = file_name.rsplit(".", 1)[0].lower().replace("-", "_").replace(" ", "_")
    full_table = f"bronze.{table_name}"

    print(f"\nProcessing {file_name}  →  {full_table}  (format: {read_format})")

    # --- Read the file ----------------------------------------------------------
    if read_format == "csv":
        # Sniff the field separator and encoding from the first line of the file
        head = dbutils.fs.head(file_path, 4096)
        first_line = head.split("\n")[0]
        sep = ";" if first_line.count(";") > first_line.count(",") else ","
        encoding = "ISO-8859-1" if "\ufffd" in head else "UTF-8"
        df = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "false")
            .option("escape", '"')
            .option("sep", sep)
            .option("encoding", encoding)
            .csv(file_path)
        )
    elif read_format == "text":
        df = spark.read.text(file_path)
    elif read_format == "xml":
        df = (
            spark.read
            .option("rowTag", "row")
            .format("xml")
            .load(file_path)
        )
    else:
        reader = spark.read.format(read_format)
        if read_format == "json":
            reader = reader.option("multiLine", "true")
        df = reader.load(file_path)

    # --- Make column names Delta-safe (no spaces, ;(){} etc.) ------------------
    df = df.toDF(*[re.sub(r"[ ,;{}()\n\t=]", "_", c) for c in df.columns])

    # --- Add bronze-layer metadata columns -------------------------------------
    df = (
             df.withColumn("_source_file", F.col("_metadata.file_name"))
               .withColumn("_source_path", F.col("_metadata.file_path"))
               .withColumn("_ingestion_ts", F.current_timestamp())
         )

    # --- Write to the bronze schema as a managed Delta table --------------------
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table)
    )

    row_count = spark.table(full_table).count()
    print(f"  ✓ Written {row_count} rows to {full_table}")

print("\n✅ All files processed.")
